# Etapa 0 — Preparação, Consolidação e Validação dos Dados
### Pipeline de ETL (Extract, Transform, Load) do Siconfi / FINBRA

Este notebook documenta de forma visual e explicada como os dados brutos de despesas das capitais brasileiras (2020 a 2025) são extraídos, limpos, consolidados, enriquecidos e carregados no banco de dados analítico (**DuckDB**) em formato otimizado (**Parquet**).

## 1. Importação das Dependências
Vamos carregar as bibliotecas padrão do Python para manipulação de arquivos, expressões regulares e processamento de dados (`pandas` e `duckdb`).

In [ ]:
import os
import zipfile
import re
import time
import logging
from pathlib import Path
import pandas as pd
import duckdb

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
pd.options.display.float_format = '{:,.2f}'.format

## 2. Configuração dos Caminhos
Definimos os caminhos para as pastas de dados compactados, dados extraídos e pastas para os arquivos consolidados.

In [ ]:
# Caminhos relativos à raiz do projeto
raiz_projeto = Path.cwd().parent
pasta_compactos = raiz_projeto / 'dados_compactos'
pasta_extraidos = raiz_projeto / 'dados_extraidos'
pasta_processados = raiz_projeto / 'data' / 'processed'
caminho_parquet = pasta_processados / 'finbra_consolidado.parquet'
caminho_duckdb = raiz_projeto / 'finbra.duckdb'

# Garantir a existência das pastas
pasta_extraidos.mkdir(parents=True, exist_ok=True)
pasta_processados.mkdir(parents=True, exist_ok=True)

print(f"Pasta compactos: {pasta_compactos.resolve()}")
print(f"Pasta extraídos: {pasta_extraidos.resolve()}")
print(f"Parquet destino: {caminho_parquet.resolve()}")
print(f"DuckDB destino: {caminho_duckdb.resolve()}")

## 3. Extração dos Arquivos ZIP (Passo 1)
Esta função varre a pasta `dados_compactos/` recursivamente em busca de arquivos `.zip`, descompactando-os para a pasta `dados_extraidos/` mantendo as pastas de origem dos anos.

In [ ]:
def descompactar_arquivos_zip(origem: Path, destino: Path):
    zips = list(origem.rglob('*.zip'))
    if not zips:
        print("Nenhum ZIP encontrado para extração.")
        return
        
    print(f"Encontrados {len(zips)} arquivos ZIP para extração.")
    for caminho_zip in zips:
        # Descobrir a subpasta (ano)
        subpasta = caminho_zip.relative_to(origem).parent
        pasta_destino_arquivo = destino / subpasta
        pasta_destino_arquivo.mkdir(parents=True, exist_ok=True)
        
        print(f"Extraindo: {caminho_zip.name} -> {pasta_destino_arquivo}")
        with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
            zip_ref.extractall(pasta_destino_arquivo)
            
descompactar_arquivos_zip(pasta_compactos, pasta_extraidos)

## 4. Leitura e Enriquecimento de Dados (Passo 2)
Para consolidar os arquivos corretamente, precisamos:
1. **Identificar o Ano**: O ano será obtido a partir da estrutura de pastas (ex: `/2022/finbra.csv` -> Ano = 2022).
2. **Ler o CSV com particularidades do Siconfi**:
   - Ignorar as 3 primeiras linhas (metadados e rótulos).
   - Utilizar o encoding `ISO-8859-1` (Latin-1) para evitar que acentos quebrem.
   - Utilizar o separador ponto e vírgula `;`.
   - Tratar a vírgula `,` como separador decimal.
3. **Enriquecer com Tipo de Conta (`ContaTipo`)**:
   - `Total_Geral`: Despesas agregadas de consolidação ('Despesas Exceto Intraorçamentárias' e 'Despesas Intraorçamentárias').
   - `Subfunção`: Formato `DD.DDD - Descrição` (ex: `10.301 - Atenção Básica`).
   - `Função`: Formato `DD - Descrição` (ex: `10 - Saúde`).
   - `Subfuncao_agregada`: Formato `FUDD - Descrição` (ex: `FU10 - Demais Subfunções`).
4. **Garantir que a coluna 'Valor' seja numérica**.

In [ ]:
def descobrir_ano_pelo_caminho(caminho: Path) -> int:
    for parte in caminho.parts:
        if re.fullmatch(r'\d{4}', parte):
            return int(parte)
    raise ValueError(f"Não foi possível identificar o ano no caminho {caminho}")

def classificar_conta(conta: str) -> str:
    if not isinstance(conta, str):
        return 'Outros'
        
    conta = conta.strip()
    
    # Totais gerais de consolidação
    if conta in ['Despesas Exceto Intraorçamentárias', 'Despesas Intraorçamentárias']:
        return 'Total_Geral'
        
    # Subfunções (DD.DDD - Texto)
    if re.match(r'^\d{2}\.\d{3}\s*-', conta):
        return 'Subfunção'
        
    # Funções (DD - Texto)
    if re.match(r'^\d{2}\s*-', conta):
        return 'Função'
        
    # Subfunções agregadas (FUDD - Texto)
    if re.match(r'^FU\d{2}\s*-', conta):
        return 'Subfuncao_agregada'
        
    return 'Outros'

## 5. Execução do Pipeline de Consolidação
Vamos carregar cada CSV, aplicar a limpeza e o enriquecimento e acumulá-los em uma única tabela consolidada.

In [ ]:
arquivos_csv = list(pasta_extraidos.rglob('*.csv'))
print(f"Encontrados {len(arquivos_csv)} arquivos CSV para processamento.")

dfs = []
for caminho_csv in arquivos_csv:
    ano = descobrir_ano_pelo_caminho(caminho_csv)
    print(f"Processando ano {ano}: {caminho_csv.name}...")
    
    # Leitura com os parâmetros do Siconfi
    df_bruto = pd.read_csv(
        caminho_csv,
        sep=';',
        skiprows=3,
        encoding='latin-1',
        decimal=','
    )
    
    # Limpeza de colunas
    df_bruto.columns = df_bruto.columns.str.strip()
    
    # Enriquecimento
    df_bruto['Ano'] = ano
    df_bruto['ContaTipo'] = df_bruto['Conta'].apply(classificar_conta)
    df_bruto['Valor'] = pd.to_numeric(df_bruto['Valor'], errors='coerce')
    
    dfs.append(df_bruto)

df_consolidado = pd.concat(dfs, ignore_index=True)
print(f"Processamento concluído. DataFrame consolidado gerado com {len(df_consolidado)} linhas.")

## 6. Validação dos Dados (Passo 2 - Validação)
Executamos testes de sanidade na base consolidada:
1. Quantidade de capitais declarantes por ano (devem ser 26 capitais, identificando se 2025 está incompleto).
2. Existência de valores nulos nas colunas críticas.
3. Validação do tipo numérico da coluna 'Valor'.

In [ ]:
print("=== VALIDAÇÃO DE COMPLETUDE POR ANO ===")
capitais_por_ano = df_consolidado.groupby('Ano')['Instituição'].nunique()
for ano, qtd in capitais_por_ano.items():
    marcador = " [!] INCOMPLETO" if qtd < 26 else " OK"
    print(f"  Ano {ano}: {qtd} capitais unicas - {marcador}")

print("\n=== VALIDAÇÃO DE VALORES NULOS ===")
nulos = df_consolidado.isnull().sum()
for col, qtd in nulos.items():
    print(f"  {col}: {qtd} nulos")

print("\n=== VALIDAÇÃO DA COLUNA VALOR ===")
is_numeric = pd.api.types.is_numeric_dtype(df_consolidado['Valor'])
print(f"  A coluna 'Valor' eh numerica? {is_numeric} (dtype: {df_consolidado['Valor'].dtype})")
if not is_numeric:
    raise TypeError("ERRO CRÍTICO: Coluna 'Valor' não é numérica!")

## 7. Persistência em Parquet (Passo 3 - Parquet)
Salvamos o DataFrame consolidado no formato colunar comprimido Parquet.

In [ ]:
df_consolidado.to_parquet(caminho_parquet, engine='pyarrow', index=False)
print(f"Base consolidada persistida com sucesso em Parquet: {caminho_parquet.resolve()}")

## 8. Carga no Banco Analítico DuckDB (Passo 3 - DuckDB)
Importamos o Parquet para o DuckDB como uma tabela física chamada `despesas_finbra`, garantindo a indexação ideal e velocidade de leitura nas análises.

In [ ]:
con = duckdb.connect(str(caminho_duckdb))

# Cria a tabela física a partir do arquivo Parquet
con.execute("DROP TABLE IF EXISTS despesas_finbra")
con.execute(f"CREATE TABLE despesas_finbra AS SELECT * FROM read_parquet('{caminho_parquet}')")

total_registros = con.execute("SELECT COUNT(*) FROM despesas_finbra").fetchone()[0]
print(f"Banco DuckDB carregado com a tabela 'despesas_finbra'. Total de registros importados: {total_registros}")

# Verificar tabelas no banco
print("\nTabelas no banco:")
print(con.execute("SHOW TABLES").df())

con.close()
print("\nConexão com DuckDB encerrada com sucesso.")

## Conclusão da Etapa 0

Com esta etapa concluída:
1. Os arquivos ZIP foram extraídos automaticamente.
2. A consolidação foi feita lidando com as nuances do formato de dados brasileiro.
3. A integridade dos dados foi validada (mostrando que 2025 está incompleto).
4. A persistência foi feita em Parquet e a tabela `despesas_finbra` foi criada fisicamente no DuckDB.

Podemos agora seguir para o notebook `01_calculo_indicadores.ipynb` para iniciar as análises orçamentárias.

In [ ]:
print('✓ Notebook 00 executado com sucesso')
print('✓ Dados brutos consolidados e salvos em Parquet/DuckDB')
print(f'✓ Registros processados: {len(df_consolidado)}')
print(f'✓ Data/hora: {pd.Timestamp.now()}')